In [1]:
from trot.config import configure_once

configure_once()

In [2]:
import jax.numpy as jnp
import numpy as np
from pyscf import gto
from pyscf import ao2mo, scf

L = 8
U = 12
t = 1.0
n = L 
mol = gto.M(verbose=4)
mol.incore_anyway = True
mol.nelec = (L//2,L//2)
inital_state = np.zeros((2,L,L))
for i in range(L//2):
    inital_state[0,2*i,2*i] = 1
    inital_state[1,2*i+1,2*i+1] = 1
print(inital_state)
h1 = np.zeros((n,n))
for i in range(n-1):
    h1[i,i+1] = h1[i+1,i] = -t
eri = np.zeros((n,n,n,n))
for i in range(n):
    eri[i,i,i,i] = U
mf = scf.UHF(mol)
mf.get_hcore = lambda *args: h1
mf.get_ovlp = lambda *args: np.eye(L)
mf._eri = ao2mo.restore(8, eri, L)
mf.kernel(inital_state)


System: uname_result(system='Darwin', node='WC292CX16G', release='25.5.0', version='Darwin Kernel Version 25.5.0: Mon Apr 27 20:38:56 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6000', machine='arm64')  Threads 1
Python 3.12.14 (main, Aug 12 2026, 13:57:54) [Clang 21.0.0 (clang-2100.1.1.101)]
numpy 2.5.3  scipy 1.18.1  h5py 3.16.0
Date: Thu Oct  1 17:56:55 2026
PySCF version 2.14.0
PySCF path  /Users/fnappi/.trot/lib/python3.12/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 0
[INPUT] num. electrons = 0
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom

nuclear repulsion = 0
number of shells = 0
number of NR pGTOs = 0
number of NR cGTOs = 0
basis = sto-3g
ecp = {}
CPU time:         1.02
[[[1. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 

Overwritten attributes  get_hcore get_ovlp  of <class 'pyscf.scf.uhf.UHF'>


np.float64(-1.1597070476816338)

In [3]:
from trot.trial.ghf import make_ghf_trial_ops
from trot.trial.ghf import GhfTrial
from trot.ham.hubbard import HamHubbard
from trot.driver import run_qmc
from trot.core.system import System
from trot.prop.types import QmcParams
from trot.meas.ghf import make_ghf_meas_ops_hubbard
from trot.prop.cpmc_slow import make_prop_ops
from trot.prop import blocks

sys = System(L,(L//2,L//2),'unrestricted')
params = QmcParams(n_walkers=200,n_blocks=500,n_eql_blocks=100,seed=2)
ham = HamHubbard(jnp.array(h1),U)
orbitals = mf.mo_coeff
nocc = L//2
trial_data = np.block([
    [orbitals[0][:, :L//2],        np.zeros((L, L//2))],
    [np.zeros((L, L//2)),          orbitals[1][:, :L//2]],
])  
trial = GhfTrial(jnp.array(trial_data))
meas_ops = make_ghf_meas_ops_hubbard(sys)
trial_ops = make_ghf_trial_ops(sys)
prop_ops = make_prop_ops(ham,'unrestricted')
prop_ctx = prop_ops.build_prop_ctx(ham, None, params)

run = run_qmc(
    sys=sys,
    params=params,
    ham_data=ham,
    trial_data=trial,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
    prop_ctx=prop_ctx,
    block_fn=blocks.block,
)

[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   -1.1597070477  2.000000e+02           0       0.0
[eql   20/100]   -1.2063800878  2.002611e+02           0       2.1
[eql   40/100]   -1.2320963894  2.003371e+02           0       4.1
[eql   60/100]   -1.2502892037  2.000307e+02           0       5.7
[eql   80/100]   -1.3159092230  2.001455e+02           0       7.3
[eql  100/100]   -1.6546258209  2.011148e+02           0       8.9

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   50/500]   -1.4652175939   1.353e-01     -1.4652175939  2.001350e+02           0      0.087      13.3
[blk  100/500]   -1.3814918158   7.475e-02     -1.2974411683  1.993614e+02           0      0.079      17.3
[blk  150/500]   -1.3531553495   5.311e-02     -1.2965301595  1.99916

In [4]:
from trot.trial.ghf import make_ghf_trial_ops
from trot.trial.ghf import GhfTrial
from trot.ham.hubbard import HamHubbard
from trot.driver import run_qmc
from trot.core.system import System
from trot.prop.types import QmcParams
from trot.meas.ghf import make_ghf_meas_ops_hubbard
from trot.prop.cpmc import make_prop_ops
from trot.prop import blocks

sys = System(L,(L//2,L//2),'unrestricted')
params = QmcParams(n_walkers=200,n_blocks=500,n_eql_blocks=100,seed=2)
ham = HamHubbard(jnp.array(h1),U)
# _,orbitals = np.linalg.eig(h1)
orbitals = mf.mo_coeff
_data = np.block([
    [ orbitals[0,:,:L//2],       orbitals[1,:,:L//2]],
    [ orbitals[0,:,:L//2],          -orbitals[1,:,:L//2]],
])  
trial = GhfTrial(jnp.array(trial_data))
meas_ops = make_ghf_meas_ops_hubbard(sys)
trial_ops = make_ghf_trial_ops(sys)
prop_ops = make_prop_ops(ham,'unrestricted',trial_ops)
prop_ctx = prop_ops.build_prop_ctx(ham, None, params)

run = run_qmc(
    sys=sys,
    params=params,
    ham_data=ham,
    trial_data=trial,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
    prop_ctx=prop_ctx,
    block_fn=blocks.block,
)

[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   -1.1597070477  2.000000e+02           0       0.0
[eql   20/100]   -1.2063800878  2.002611e+02           0       1.3
[eql   40/100]   -1.2320963894  2.003371e+02           0       2.4
[eql   60/100]   -1.2502892037  2.000307e+02           0       3.2
[eql   80/100]   -1.3159092230  2.001455e+02           0       4.0
[eql  100/100]   -1.6546258209  2.011148e+02           0       4.8

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   50/500]   -1.4652175939   1.353e-01     -1.4652175939  2.001350e+02           0      0.048       7.2
[blk  100/500]   -1.3814918158   7.475e-02     -1.2974411683  1.993614e+02           0      0.039       9.1
[blk  150/500]   -1.3531553495   5.311e-02     -1.2965301595  1.99916

In [6]:
from trot.gmps.driver import make_mps_cpmc_ops
from trot.gmps.utils import sd_to_gmps
from trot.trial.mps import make_mps_trial
from trot.prop.types import QmcParamsMps

Ca, Cb = mf.mo_coeff[0][:, :L//2], mf.mo_coeff[1][:, :L//2]
ghf = GhfTrial(jnp.array(np.block([[Ca, Cb], [Ca, -Cb]])))
tensors = sd_to_gmps(Ca, Cb, mode="maximal").tensors
R = np.array([[1.0, 1.0], [1.0, -1.0]]) / np.sqrt(2.0)
M = np.zeros((4, 4))
M[0, 0] = 1.0
M[1:3, 1:3] = R
M[3, 3] = -1
new_tensors = [np.einsum("ps,asb->apb", M, np.asarray(A, dtype=float)) for A in tensors]
trial = make_mps_trial(new_tensors,nelec=(L//2, L//2))
params = QmcParamsMps(n_walkers=200, n_blocks=500, n_eql_blocks=100, seed=2,
                      orbital_plan="maximal", walker_channel_chi=None)
ops = make_mps_cpmc_ops(ham, trial, sys, params)
run = run_qmc(sys=sys, params=params, ham_data=ham, trial_data=trial, trial_ops=ops.trial_ops,
              meas_ops=ops.meas_ops, prop_ops=ops.prop_ops,
              prop_ctx=ops.prop_ops.build_prop_ctx(ham, None, params), block_fn=blocks.block)


[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   -1.7407111942  2.000000e+02           0       0.0


KeyboardInterrupt: 